# Soma — free Colab (T4) run

**What this does:** runs Meta's public **TRIBE v2** brain-encoding model on a folder of
video clips and caches, per clip, the predicted cortical activation
(`preds_<id>.npy`, shape `(n_seconds, 20484)`) plus a per-second arc
(`arc_<id>.csv` + `arc_<id>.json`) that is drop-in compatible with
`tvsum_prep.py` / `honest_corr_timeseries.py` / the demo player.

**Config:** audio + video only (the text / LLaMA path is OFF) — the verified config that
ran on a free T4 before.

**Speed (built in):** dataloader workers are capped at 2 (the shipped default is 20, which thrashes Colab's 2 cores — the prime cause of a marathon run). This changes **no numbers**. On Colab **Pro, pick an L4 GPU** (Runtime ▸ Change runtime type): the video encoder is the heavy part and L4 is much faster than T4 for the identical output.

**Honesty (do not edit these out):**
- Step 1 — *video → brain activation* — is the only validated link (TRIBE, benchmarked vs real fMRI).
- The activation → attention/engagement reading is **our downstream hypothesis, not a result.**
- `preds` are **z-scored, SIGNED BOLD (~[-1, +1])**, NOT 0..1 probabilities. Cell 4 prints
  the real distribution once so you can confirm this — never assume it.
- Nothing here is trained on our side; this is inference only. No fabricated numbers.

**Two ways to feed it data — pick ONE:**
- **Path A — simplest, heaviest on GPU:** Cell 2 downloads all of TVSum into Colab and does everything here.
- **Path B — recommended, least GPU:** run a trim/prep tool on your laptop (`tvsum_trim.py`, `cognimuse_films.py`, `veatic_prep.py --stage-clips`, `studyforrest_film.py`) — each trims + downscales clips for free — drop the small clips in Google Drive, and use **Cell 2B**. Colab does *only* the brain-math, and results cache to Drive so runs **resume** across disconnects — a wiped machine or a hit quota never costs you a redo.
  - **Batches:** upload each batch into its own Drive subfolder (`clips/cognimuse`, `clips/cognimuse_batch2`, `clips/veatic`, …). Cell 2B sweeps them all and Cell 4 scores only the not-yet-done clips, so you can keep adding batches on different nights.

**Run order (attention lane · AV · free T4/L4):** Cell 1 (install → **restart runtime**) → **Cell 2 _or_ Cell 2B** → **Cell 2C** (build the DAN + LANGUAGE masks) → Cell 3 (load model) → Cell 4 (extract) → Cell 5 (download). *(The message lane and the exact TRIMODAL order are in the next cell.)*

## Two lanes, two run orders — read this before you run

This notebook writes **two predicted product lanes** per clip, as columns in `arc_<id>.csv`
(alongside `global_mag`, the whole-cortex documented likely-NULL baseline):

| lane | column | a-priori mask (`build_roi_mask.py --network …`) | what it is (a **HYPOTHESIS**, never a measured result) | needs |
|---|---|---|---|---|
| **Attention** | `dan_mag` | `dan` — dorsal-attention network | predicted top-down **attention load** (the TVSum-proxy family — importance ≠ retention) | **AV** (audio+video) is enough → free **T4/L4** |
| **Message** | `language_mag` | `language` — language / semantic-association network | predicted **semantic-integration load** — *never* "the viewer understood it" (no comprehension validation exists yet) | **TRIMODAL** (audio+video+**text**); AV fills the column but with no text branch it is a **plumbing/shape demo, not** the message signal |

Every `arc_<id>.csv` is therefore `t_sec,global_mag,dan_mag,language_mag` — the exact schema the
within-item variant ranker (`compare_cuts.py`) consumes.

**Honesty (do not soften):** both lanes are **predicted** arcs, badged hypotheses. "Compare Your
Cuts" ranks a brand's **own** variants *relative* to each other — it is **never** an absolute
"this ad will win" score (that between-item axis tested null).

### Run order (a) — ATTENTION lane · AV · free T4/L4
**Cell 1** (install → **restart**) → **Cell 2** *or* **Cell 2B** (stage clips) → **Cell 2C**
(build/load the DAN + LANGUAGE masks) → **Cell 3** (load AV model) → **Cell 4** (extract → `arcs/`)
→ **Cell 5** (download). *(`language_mag` is written but is a shape demo here — no text branch drives it.)*

### Run order (b) — ATTENTION + MESSAGE lanes · TRIMODAL · needs A100
**Cell 1** (install → **restart**) → **Cell T1** (trimodal deps + gated `meta-llama/Llama-3.2-3B`
login; wait for `[OK]`) → **Cell 2B** (mount Drive + clips) → **Cell 2C** (masks) → **Cell T3**
(load trimodal model) → **Cell T4** (extract → `arcs_trimodal/`). *(Skip Cell 2 and Cells 3–4 —
those are the AV path.)* Requires an **A100 + High-RAM**, gated **LLaMA-3.2-3B**, and **whisperx**
(Cell T1 installs it). This is the run where `language_mag` becomes the real message lane.


## Cell 1 — install pinned deps  (then Runtime ▸ Restart session)

In [ ]:
# === Cell 1: install pinned deps for the verified audio+video TRIBE v2 stack =====
# After this finishes it STOPS with a SystemExit telling you to RESTART THE RUNTIME
# (Runtime > Restart session), then run again starting at Cell 2 — skip Cell 1 on the
# second pass. The restart is REQUIRED: NumPy/SciPy are force-pinned below into TRIBE's
# known-good range and Colab must reload them. Do not skip it.
import os, sys, subprocess
from importlib import metadata

# facebook/tribev2 is ~1 GB; a cold T4 download can be slow — give HF room to breathe.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

def _mm(v):
    out = []
    for tok in v.split("."):
        d = "".join(c for c in tok if c.isdigit())
        if not d:
            break
        out.append(int(d))
        if len(out) == 2:
            break
    while len(out) < 2:
        out.append(0)
    return tuple(out)

need = False
try:
    if _mm(metadata.version("numpy")) >= (2, 1):   # TRIBE needs numpy>=1.26,<2.1
        need = True
except Exception:
    need = True
try:
    if not ((1, 13) <= _mm(metadata.version("scipy")) < (1, 16)):
        need = True
except Exception:
    need = True
for dist in ["tribev2", "neuralset", "exca", "nibabel", "nilearn"]:
    try:
        metadata.version(dist)
    except metadata.PackageNotFoundError:
        need = True

if need:
    print("Installing Soma / TRIBE deps (a few minutes on a cold T4)...")
    # Full notebook stack first...
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
         "exca", "yt-dlp", "pillow", "pandas", "matplotlib", "moviepy"],
        check=True,
    )
    # ...then force NumPy/SciPy back to TRIBE's compiled-dep range.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--force-reinstall",
         "numpy>=1.26.4,<2.1", "scipy>=1.13,<1.16"],
        check=True,
    )
    # keep torchaudio in LOCKSTEP with torch (the tribev2 install can pull a
    # newer torchaudio -> "undefined symbol: aoti_torch_abi_version" crash in Cell 4).
    # Read the freshly-installed torch from a subprocess (this process has a stale one).
    _tv = subprocess.run([sys.executable, "-c",
        "import torch;print(torch.__version__.split('+')[0])"],
        capture_output=True, text=True).stdout.strip()
    _cu = subprocess.run([sys.executable, "-c",
        "import torch;print('cu'+(torch.version.cuda or '12.1').replace('.',''))"],
        capture_output=True, text=True).stdout.strip()
    if _tv:
        # CLEAN reinstall (uninstall first): a --force-reinstall can leave mixed
        # torchaudio files -> "cannot import name '_init_sox'". Remove, then install one copy.
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=False)
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             f"torchaudio=={_tv}", "--index-url", f"https://download.pytorch.org/whl/{_cu}"],
            check=False,
        )
    raise SystemExit(
        "Install complete. Now: Runtime > Restart session, then run from Cell 2 (skip Cell 1)."
    )

print("Deps already present.")
print("  NumPy:", metadata.version("numpy"), " SciPy:", metadata.version("scipy"))
print("  HF_HUB_DOWNLOAD_TIMEOUT =", os.environ["HF_HUB_DOWNLOAD_TIMEOUT"])


## Cell 2 — get the data (TVSum50, no Google Drive needed)

Downloads the official TVSum50 archive **straight into this Colab runtime**, extracts it,
and points the pipeline at a subset of videos. It also tucks `ydata-tvsum50.mat` into the
results folder so it ends up in the download zip (you need it for `tvsum_prep.py` on your
laptop). Using your **own ad clips** instead? Skip this and run **Cell 2-ALT** below.

> TVSum videos are full-length (1–11 min each), so extraction is where the time goes.
> Start with a small `N_CLIPS`; longer videos = more timepoints = a *stronger* per-video test.

In [ ]:
# === Cell 2: download + stage TVSum50 (no Drive needed) =========================
import subprocess, shutil
from pathlib import Path

N_CLIPS = 3    # first run: prove the pipeline end-to-end fast, then raise to 10+ for a real batch

WORK = Path("/content/tvsum"); WORK.mkdir(exist_ok=True)
TGZ  = WORK / "tvsum50_ver_1_1.tgz"
URL  = "http://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz"

if not TGZ.exists():
    print("Downloading TVSum50 (641 MB) ...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(TGZ), URL], check=True)
if not any(WORK.rglob("*.mp4")):
    print("Extracting ...")
    subprocess.run(["tar", "xzf", str(TGZ), "-C", str(WORK)], check=True)
# TVSum nests the real data inside .zip files — unzip those too
for _z in sorted(WORK.rglob("*.zip")):
    subprocess.run(["unzip", "-o", "-q", str(_z), "-d", str(_z.parent)], check=False)

# locate videos under ANY common extension (TVSum may not be .mp4); if none,
# print the extracted tree so the layout is obvious instead of a blind crash.
import os, collections
VIDEO_EXTS = (".mp4",".webm",".mkv",".avi",".mov",".m4v",".flv",".mpg",".mpeg")
mp4s = sorted(p for p in WORK.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
mats = sorted(WORK.rglob("ydata-tvsum50.mat"))
if not mp4s:
    print("No video files found. What actually extracted:")
    for r,_,f in os.walk(WORK):
        d = r.replace(str(WORK),"").count(os.sep)
        if d <= 2: print("  "*d + os.path.basename(r) + f"/  ({len(f)} files)")
    exts = collections.Counter(os.path.splitext(f)[1].lower()
                               for r,_,fs in os.walk(WORK) for f in fs)
    print("file extensions present:", exts.most_common(20))
    raise SystemExit("No videos in the archive — paste the tree above to Claude "
                     "(we may need yt-dlp to fetch them by YouTube ID).")
VIDEO_SRC = mp4s[0].parent
MAT = mats[0] if mats else None

# ---- config the rest of the notebook reads (drop-in for the Drive cell) ---------
CLIP_DIR = WORK / "clips"; CLIP_DIR.mkdir(exist_ok=True)
for v in mp4s[:N_CLIPS]:
    dst = CLIP_DIR / v.name
    if not dst.exists():
        shutil.copy(v, dst)

OUT_DIR = WORK / "arcs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
GLOB = "*" + mp4s[0].suffix.lower()
ROI_MASK_PATH = None   # legacy single-mask hook; Cell 2C (run it next) builds + sets
                       # ROI_MASKS = {"dan": ..., "language": ...} -> the two product lanes.
DEMO_FEATURE = "roi"   # Cell 2C overrides this to "dan" (attention lane); else "global".

# stash the annotations so Cell 5's zip carries them back to your laptop
if MAT is not None:
    shutil.copy(MAT, OUT_DIR / "ydata-tvsum50.mat")

clips = sorted(CLIP_DIR.glob(GLOB))
print(f"\nTVSum ready:")
print(f"  videos in archive : {len(mp4s)}  (processing {len(clips)} this run — edit N_CLIPS)")
print(f"  CLIP_DIR : {CLIP_DIR}")
print(f"  OUT_DIR  : {OUT_DIR}")
print(f"  .mat     : {MAT}  (copied into OUT_DIR -> lands in the results zip)")
for c in clips:
    print("   -", c.name)

## Cell 2B — LIGHT path: trimmed clips from Drive (recommended, least GPU)

**Use this INSTEAD of Cell 2** once you've prepped clips on your laptop with
`tvsum_trim.py` (it trims to ~2 min + downscales) — or for your own ad clips.
Colab does only the GPU brain-math: no 641 MB download, no extraction, no ffmpeg on
the GPU clock. And because `OUT_DIR` lives on Drive, **finished clips persist** — a
disconnect or a hit quota resumes instead of restarting.

In [ ]:
# === Cell 2B — LIGHT path: pre-trimmed clips from Drive (RECOMMENDED) ============
# Pair with the laptop-side trim/prep tools (tvsum_trim.py, cognimuse_films.py,
# veatic_prep.py --stage-clips, ...): they trim + downscale clips locally (free), and you
# drop the small clips into a Drive folder below.
#
# BATCH-FRIENDLY (new): upload clips in BATCHES, each in its OWN subfolder
# (clips/cognimuse, clips/cognimuse_batch2, clips/veatic, ...). This cell sweeps them all,
# so adding a new batch never disturbs the ones already uploaded or already scored.
#
# RESUME: OUT_DIR is on Drive, so finished clips survive disconnects. Re-running Cell 4
# SKIPS clips already done (a preds_<id>.npy exists) -> today's batch + tomorrow's batch
# just stack up, and a mid-batch disconnect never costs a redo. Furthest free-quota reach.
# (SKIP Cell 2 above if you use this cell.)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

# ---- EDIT THESE to match your Drive layout --------------------------------------
# One folder OR several. With RECURSIVE=True each entry also sweeps its subfolders, so a
# single "clips" root with per-dataset subfolders is all you need; list extra top-level
# folders here only if they live outside it.
CLIP_DIRS = [
    Path("/content/drive/MyDrive/soma/clips/ads"),   # ADS-ONLY scope (was .../clips) — only the ad batch
    # Path("/content/drive/MyDrive/soma/clips_extra"),   # e.g. a second top-level folder
]
RECURSIVE = True          # also pick up *.mp4 inside subfolders of each CLIP_DIRS entry
OUT_DIR   = Path("/content/drive/MyDrive/soma/arcs_ads")   # ADS outputs isolated (was .../arcs); resume-safe
GLOB      = "*.mp4"                                     # which files to process

# LANES: Cell 2C (run right after this cell) builds BOTH the DAN (attention) and the
# LANGUAGE (message) mask and sets ROI_MASKS -> arc_<id>.csv gets dan_mag + language_mag.
# The single ROI_MASK_PATH below is a LEGACY hook (a lone DMN mask); it is harmless if the
# file isn't there, because Cell 2C overrides it. GLOBAL alone is the documented NULL baseline.
ROI_MASK_PATH = Path("/content/drive/MyDrive/soma/roi_mask_dmn.npy")
DEMO_FEATURE  = "roi"   # Cell 2C overrides -> "dan"; falls back to "global" if no lane mask.
# ---------------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the ROI mask robustly. Two real failure modes we guard against here so the run
# can't silently fall back to GLOBAL-only (the gap that made the first real run a null
# baseline): (1) the path doesn't exist; (2) a Google Drive upload NESTED the file inside a
# folder of the SAME name (roi_mask_dmn.npy/roi_mask_dmn.npy) -> the path is a DIRECTORY,
# which .exists() passes but np.load() later crashes on (IsADirectoryError). Accept a real
# file, else descend into a same-named folder to find the .npy, else warn LOUDLY.
def _resolve_mask(p):
    if p is None:
        return None
    p = Path(p)
    if p.is_file():
        return p
    if p.is_dir():
        inner = sorted(p.rglob("*.npy"))
        if inner:
            print(f"[roi] note: {p} is a folder (Drive nested the upload); using {inner[0]}")
            return inner[0]
    return None

ROI_MASK_PATH = _resolve_mask(ROI_MASK_PATH)
if ROI_MASK_PATH is None:
    print("!! ROI mask NOT found as a real .npy file.")
    print("   -> you'll get GLOBAL ONLY (the documented null baseline), NOT the open ROI test.")
    print("   Upload roi_mask_dmn.npy to /content/drive/MyDrive/soma/ (as a FILE, not a folder),")
    print("   set ROI_MASK_PATH above, and re-run this cell.\n")
else:
    print("[roi] mask ->", ROI_MASK_PATH, "-> arcs will include roi_mag (the open test)\n")

# ---- discover clips across ALL CLIP_DIRS (dedup by stem) ------------------------
# preds/arcs are keyed by the file STEM (its <id>), so two files sharing a stem — e.g. the
# same clip uploaded into two folders — would COLLIDE: one silently wins, the other never
# runs. Keep the FIRST (deterministic, sorted) and WARN loudly, listing every collision.
def _discover(dirs, glob, recursive):
    picked, order, dups, scanned = {}, [], [], []
    for d in dirs:
        d = Path(d)
        if not d.exists():
            scanned.append((d, None))
            continue
        files = sorted(d.rglob(glob) if recursive else d.glob(glob))
        scanned.append((d, len(files)))
        for f in files:
            if f.stem in picked:
                dups.append((f.stem, picked[f.stem], f))
                continue
            picked[f.stem] = f
            order.append(f)
    return order, dups, scanned

clips, dup_stems, _scanned = _discover(CLIP_DIRS, GLOB, RECURSIVE)
CLIP_DIR = CLIP_DIRS[0]          # kept so Cell 4's guard/messages (which read CLIP_DIR) work
done = {p.stem[len("preds_"):] for p in OUT_DIR.glob("preds_*.npy")}
todo = [c for c in clips if c.stem not in done]

print("clip folders scanned:")
for d, n in _scanned:
    print(f"   - {d}  ->  {'MISSING (skipped)' if n is None else f'{n} match {GLOB!r}'}")
print(f"total    : {len(clips)} unique clip(s)  |  {len(done)} already done -> {len(todo)} to run")
if dup_stems:
    print(f"!! {len(dup_stems)} DUPLICATE stem(s): the same <id> in two folders. Keeping the")
    print("   first, skipping the rest (preds are keyed by <id>). Remove one of each pair:")
    for stem, kept, skipped in dup_stems:
        print(f"     - {stem}: keep {kept}  |  SKIP {skipped}")
for c in clips[:16]:
    print(f"   - [{'done' if c.stem in done else 'todo'}] {c.name}")
if len(clips) > 16:
    print(f"   ... (+{len(clips) - 16} more)")
if not clips:
    print("!! No clips found — run a trim/prep tool on your laptop and upload clips into one")
    print("   of the folders above (subfolders OK with RECURSIVE=True).")
print(f"out dir  : {OUT_DIR}  (on Drive -> results survive disconnects)")
print(f"roi mask : {ROI_MASK_PATH}")


## Cell 2C — build (or load) the DAN + LANGUAGE ROI masks

Run this **after Cell 2 / 2B** and **before Cell 3 / T3**. It produces the two a-priori masks the
lanes read:
- `roi_mask_dan.npy` — dorsal-attention network → **attention** lane (`dan_mag`)
- `roi_mask_language.npy` — language / semantic-association network → **message** lane (`language_mag`)

It mirrors `build_roi_mask.py` (`--network dan` / `--network language`) **inline** (Destrieux
surface atlas via nilearn) so Colab needs no repo clone. If a `roi_mask_<net>.npy` already sits
next to your outputs (e.g. you uploaded a prebuilt one to Drive), it is **loaded, not rebuilt**.
Sets `ROI_MASKS` so Cell 4 / T4 write both columns.


In [ ]:
# === Cell 2C: build/load the DAN (attention) + LANGUAGE (message) ROI masks ======
# Mirrors build_roi_mask.py's Destrieux SURFACE path (--network dan / --network language)
# INLINE, so Colab needs no repo clone. Masks are saved next to your OUT_DIR (on Drive when
# Cell 2B mounted it -> they persist and can be reused / re-uploaded). If roi_mask_<net>.npy
# already exists there it is LOADED, not rebuilt.
from pathlib import Path
import numpy as np

if "OUT_DIR" not in globals():
    raise SystemExit("→ Run Cell 2 (TVSum) or Cell 2B (Drive clips) first — it sets OUT_DIR.")

# nilearn ships with the tribev2[plotting] install; make sure it's importable.
try:
    from nilearn import datasets
except Exception:
    import subprocess, sys
    print("[setup] installing nilearn (one-time) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nilearn"], check=False)
    from nilearn import datasets

FSAVERAGE5_N = 20484   # cortical surface vertices ([lh; rh], 10242/hemi)

# a-priori region substrings — COPIED VERBATIM from build_roi_mask.py so the notebook and
# the .py agree. DAN = the attention lane; LANGUAGE = the message lane. Change these ONCE,
# record in PREREGISTRATION.md, never after seeing results.
NETWORK_REGIONS = {
    "dan": [                            # dorsal-attention network (ATTENTION lane)
        "S_intrapariet_and_P_trans",    # intraparietal sulcus
        "G_precentral",                 # frontal eye fields vicinity (precentral)
        "S_precentral-sup-part",
        "G_parietal_sup",
    ],
    "language": [                       # language / semantic-association network (MESSAGE lane)
        "G_front_inf-Triangul",         # IFG pars triangularis (Broca)
        "G_front_inf-Opercular",        # IFG pars opercularis (Broca)
        "Pole_temporal",                # temporal pole / anterior temporal lobe
        "S_temporal_sup",               # superior temporal sulcus (lexico-semantic)
        "G_temporal_middle",            # posterior middle temporal gyrus
        "G_pariet_inf-Angular",         # angular gyrus (semantic-integration hub)
    ],
}


def _build_surface_mask(regions):
    """fsaverage5 SURFACE (20484) bool mask via Destrieux (== build_roi_mask.build_mask_surface)."""
    atlas = datasets.fetch_atlas_surf_destrieux()
    labels = [l.decode() if isinstance(l, bytes) else str(l) for l in atlas["labels"]]

    def hemi(map_arr):
        m = np.zeros(len(map_arr), bool)
        for i, name in enumerate(labels):
            if any(r.lower() in name.lower() for r in regions):
                m |= (np.asarray(map_arr) == i)
        return m

    mask = np.concatenate([hemi(atlas["map_left"]), hemi(atlas["map_right"])])  # [lh; rh]
    if mask.shape[0] != FSAVERAGE5_N:
        raise SystemExit(f"mask length {mask.shape[0]} != {FSAVERAGE5_N} — check nilearn atlas res")
    if mask.sum() == 0:
        raise SystemExit("ROI matched 0 vertices — Destrieux substrings didn't hit; print atlas['labels'].")
    return mask


MASK_DIR = OUT_DIR.parent                # Drive soma root under Cell 2B; local WORK under Cell 2
MASK_DIR.mkdir(parents=True, exist_ok=True)
ROI_MASKS = {}
for net, regions in NETWORK_REGIONS.items():
    p = MASK_DIR / f"roi_mask_{net}.npy"
    if p.is_file():
        m = np.asarray(np.load(p), bool)
        print(f"[mask] {net:9s} loaded {p.name}: {int(m.sum())}/{m.shape[0]} vertices")
    else:
        print(f"[mask] {net:9s} building from Destrieux (first run downloads the atlas) ...")
        m = _build_surface_mask(regions)
        np.save(p, m)
        print(f"       wrote {p}  ({int(m.sum())}/{m.shape[0]} vertices, {m.mean():.1%} of cortex)")
    ROI_MASKS[net] = p

# lanes Cell 4 / T4 write, in column order -> arc_<id>.csv = t_sec,global_mag,dan_mag,language_mag
DEMO_FEATURE = "dan"                      # ATTENTION drives arc_<id>.json (valid on AV runs);
                                          # set = "language" if you want the trimodal message demo.
ROI_MASK_PATH = ROI_MASKS["dan"]         # back-compat for any cell/consumer that reads one mask

print("\n[lanes] arc columns -> t_sec,global_mag," + ",".join(f"{n}_mag" for n in ROI_MASKS))
print("[honesty] dan_mag      = predicted ATTENTION load (hypothesis; importance != retention).")
print("[honesty] language_mag = predicted SEMANTIC-INTEGRATION load (hypothesis; NEVER 'understood').")
print("          On AV-only runs language_mag is a plumbing/shape demo — the MESSAGE lane needs the")
print("          TRIMODAL text branch (Cell T1/T3/T4). Rank a brand's OWN cuts, relatively.")


## Cell 3 — load TRIBE v2 (audio + video, the verified config)

In [ ]:
# === Cell 3: load the model (with the VERIFIED speed fix) ========================
import os
# Re-set here too: env vars are cleared by the runtime restart after Cell 1.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")

from pathlib import Path
from tribev2.demo_utils import TribeModel

# --- SPEED FIX (verified in the TRIBE source; changes NO numbers) ----------------
# The shipped config bakes data.num_workers = N_CPUS = 20 (grids/defaults.py). On a
# free/Pro 2-core Colab box, 20 dataloader workers THRASH the CPU -- the prime suspect
# behind the ~15h run. Forcing num_workers=2 is PURELY a scheduling change: predict()
# runs the frozen model in eval mode with shuffle=False, so the (n_sec, 20484) preds are
# BIT-IDENTICAL regardless of worker count. We do NOT touch fps / resolution / chunk size
# / precision -- those WOULD change the one validated number (kept frozen).
NUM_WORKERS = 2   # matches Colab's 2 cores; use 0 for fully synchronous loading

# Cache on Drive if it is mounted (Cell 2B) so the ~1 GB weights + feature cache SURVIVE a
# disconnect and never re-download; else fall back to the (wiped-on-disconnect) local dir.
CACHE_FOLDER = (Path("/content/drive/MyDrive/soma/cache")
                if Path("/content/drive/MyDrive").exists() else Path("./cache"))
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"cache_folder = {CACHE_FOLDER}"
      + ("  (on Drive -> weights persist across disconnects)"
         if str(CACHE_FOLDER).startswith("/content/drive")
         else "  (local -> wiped on disconnect; run Cell 2B to cache on Drive)"))

print("Loading facebook/tribev2 (audio+video) -- first run downloads ~1 GB ...")
model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={"data": {"features_to_use": ["audio", "video"],
                            "num_workers": NUM_WORKERS}},
)

# Belt-and-suspenders guard: a mistyped config key would already RAISE here (the Data model
# uses extra='forbid', so a silent no-op is impossible), but we ALSO cap every DataLoader
# the run builds -- independent of any exca/config version. This physically cannot no-op.
import torch
_orig_dl_init = torch.utils.data.DataLoader.__init__
_dl_notified = {"done": False}
def _capped_dl_init(self, *a, num_workers=0, **k):
    num_workers = min(num_workers or 0, NUM_WORKERS)   # 'or 0' also handles num_workers=None
    _orig_dl_init(self, *a, num_workers=num_workers, **k)
    if not _dl_notified["done"]:
        print(f"[speed] DataLoader workers capped at {self.num_workers} (shipped default was 20).")
        _dl_notified["done"] = True
torch.utils.data.DataLoader.__init__ = _capped_dl_init

print(f"Model loaded.  features_to_use = ['audio','video']  (LLaMA/text OFF)  "
      f"num_workers -> {NUM_WORKERS}.")


## Cell 4 — batch extract

Per clip: build events → `model.predict` → save `preds_<id>.npy`, then reduce to
`arc_<id>.csv` (`t_sec, global_mag[, roi_mag]`) + `arc_<id>.json`.
The functions below mirror `batch_extract.py` so the outputs are byte-compatible with the
CPU analysis + demo. Skips already-cached clips; one bad clip cannot kill the batch.

In [ ]:
# === Cell 4: predict -> preds_<id>.npy + arc_<id>.csv + arc_<id>.json ============
# Mirrors batch_extract.py (the GPU-box runbook) so Colab output is drop-in compatible
# with tvsum_prep.py / honest_corr_timeseries.py / the demo player.
import json, time, traceback
import numpy as np
import pandas as pd
from neuralset.events.utils import standardize_events
from neuralset.events.transforms import ExtractAudioFromVideo, ChunkEvents

# --- guard: Cell 2 (or 2B) must have run to define the paths this cell needs
_need = [v for v in ("CLIP_DIR", "OUT_DIR", "GLOB", "ROI_MASK_PATH", "DEMO_FEATURE",
                     "clips") if v not in globals()]
if _need:
    raise SystemExit("→ Run Cell 2 (get TVSum) first — or Cell 2B for clips from "
                     "Drive — it sets the paths this cell needs. Missing: " + ", ".join(_need))
if not clips:
    raise SystemExit(f"0 clips found in {CLIP_DIR} (GLOB={GLOB!r}). If you meant TVSum, run "
                     "Cell 2 (the download cell); for the light path run tvsum_trim.py on your "
                     "laptop and upload the clips to the Cell 2B Drive folder.")


def build_events(video_path):
    """The working manual event-build path from the notebook (no transcription)."""
    transforms = [
        ExtractAudioFromVideo(),
        ChunkEvents(event_type_to_chunk="Audio", max_duration=60, min_duration=30),
        ChunkEvents(event_type_to_chunk="Video", max_duration=60, min_duration=30),
    ]
    initial = {"type": "Video", "filepath": str(video_path), "start": 0,
               "timeline": "default", "subject": "default"}
    df = standardize_events(pd.DataFrame([initial]))
    for t in transforms:
        df = t(df)
    return standardize_events(df)


def describe_preds(preds, tag=""):
    """Print the distribution ONCE so the z-scored/signed scale is confirmed, not assumed."""
    p = np.asarray(preds, float)
    pct = np.percentile(p, [1, 50, 90, 99])
    print(f"  [preds {tag}] shape={p.shape} min={p.min():.3f} max={p.max():.3f} "
          f"mean={p.mean():.3f} std={p.std():.3f}")
    print(f"           pct(1/50/90/99)={pct[0]:.3f}/{pct[1]:.3f}/{pct[2]:.3f}/{pct[3]:.3f}"
          f"  frac<0={np.mean(p < 0):.2%}  frac>0.6={np.mean(p > 0.6):.2%}")
    if p.min() >= 0 and p.max() <= 1:
        print("           WARNING: values look bounded [0,1] - verify this really is "
              "z-scored BOLD, not a probability, before trusting any threshold.")


def arc_from_preds(preds, roi_mask=None):
    """(T, 20484) -> per-second magnitude scalars. |preds| avoids the old threshold bug."""
    p = np.abs(np.asarray(preds, float))
    global_mag = p.mean(axis=1)
    roi_mag = None
    if roi_mask is not None:
        m = np.asarray(roi_mask, bool)
        if m.shape[0] != p.shape[1]:
            raise ValueError(f"ROI mask length {m.shape[0]} != n_vertices {p.shape[1]}")
        roi_mag = p[:, m].mean(axis=1)
    return global_mag, roi_mag


def arc_from_preds_multi(preds, lane_masks):
    """(T, 20484) -> global_mag + {lane: mag}. Same |preds| magnitude arithmetic as
    arc_from_preds, one column per a-priori lane mask (the z-BOLD scale bug can't resurface)."""
    p = np.abs(np.asarray(preds, float))
    global_mag = p.mean(axis=1)
    lanes = {}
    for name, m in lane_masks.items():
        m = np.asarray(m, bool)
        if m.shape[0] != p.shape[1]:
            raise ValueError(f"lane {name!r} mask length {m.shape[0]} != n_vertices {p.shape[1]}")
        lanes[name] = p[:, m].mean(axis=1)
    return global_mag, lanes


def detect_weak_spots(arc, fps=1.0, min_len_sec=3, drop_pctl=25):
    """A weak spot = a sustained run in the arc's own bottom quartile. A PREDICTION, not behavior."""
    a = np.asarray(arc, float)
    if len(a) < 5:
        return []
    k = max(1, int(round(fps)))
    sm = np.convolve(a, np.ones(k) / k, mode="same")
    thr = np.percentile(sm, drop_pctl)
    low = sm <= thr
    spots, i, n = [], 0, len(a)
    min_len = int(round(min_len_sec * fps))
    while i < n:
        if low[i]:
            j = i
            while j < n and low[j]:
                j += 1
            if (j - i) >= min_len:
                spots.append({"start": round(i / fps, 2), "end": round(j / fps, 2),
                              "label": f"predicted dip - {round((j - i) / fps)}s "
                                       f"low-activation stretch"})
            i = j
        else:
            i += 1
    return spots


def write_demo_json(out_dir, vid, arc, weak_spots, feature_name, fps=1.0):
    a = np.asarray(arc, float)
    lo, hi = np.percentile(a, 2), np.percentile(a, 98)
    norm = np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
    payload = {
        "video_id": vid, "fps_arc": fps, "duration_sec": round(len(a) / fps, 2),
        "feature": feature_name, "precomputed": True,
        "claim": {
            "validated": "video -> brain activation (Meta TRIBE v2, public model, "
                         "benchmarked vs real fMRI)",
            "hypothesis": "activation -> engagement (our downstream inference; "
                          "tested, not asserted)",
        },
        "timestamps": [round(i / fps, 2) for i in range(len(a))],
        "activation": [round(float(x), 4) for x in norm],
        "weak_spots": weak_spots,
    }
    path = out_dir / f"arc_{vid}.json"
    path.write_text(json.dumps(payload, indent=2))
    return path


# --- load the LANE masks (Cell 2C sets ROI_MASKS = {"dan":..,"language":..}) ------
# Each lane becomes a <name>_mag column in arc_<id>.csv, in dict order. Back-compat: if
# ROI_MASKS is absent we fall back to the single ROI_MASK_PATH -> a lone 'roi' column.
LANE_MASKS = {}
_lane_paths = {}
if "ROI_MASKS" in globals() and ROI_MASKS:
    _lane_paths = dict(ROI_MASKS)
elif "ROI_MASK_PATH" in globals() and ROI_MASK_PATH is not None:
    _lane_paths = {"roi": ROI_MASK_PATH}
for _name, _p in _lane_paths.items():
    _m = np.asarray(np.load(_p), bool)
    LANE_MASKS[_name] = _m
    print(f"[lane] {_name:9s} {Path(_p).name}: {int(_m.sum())}/{_m.shape[0]} vertices")
if not LANE_MASKS:
    print("[note] no lane masks -> GLOBAL only. Run Cell 2C to build the DAN + LANGUAGE masks.")
else:
    print("[lanes] arc columns -> t_sec,global_mag," + ",".join(f"{n}_mag" for n in LANE_MASKS))

printed_scale = False
n_ok, n_fail = 0, 0
clip_secs = []            # wall-clock per freshly-predicted clip -> live ETA (no more mystery runs)
_todo = sum(1 for vp in clips if not (OUT_DIR / f"preds_{vp.stem}.npy").exists())
_left = _todo
for vp in clips:
    vid = vp.stem
    npy = OUT_DIR / f"preds_{vid}.npy"
    try:
        if npy.exists():
            print(f"[skip-cached] {vid}")
            preds = np.load(npy)
        else:
            print(f"[predict] {vid} ... ({_left} of {_todo} to go)")
            _t0 = time.time()
            events = build_events(vp)
            preds, _segments = model.predict(events=events)
            preds = np.asarray(preds, float)
            np.save(npy, preds.astype(np.float32))  # float32 halves Drive size; z-scored BOLD needs no more precision
            _dt = time.time() - _t0; clip_secs.append(_dt); _left -= 1
            _avg = sum(clip_secs) / len(clip_secs)
            print(f"    took {_dt:.0f}s  |  avg {_avg:.0f}s/clip  |  ETA remaining "
                  f"{_left}: ~{_avg * _left / 60:.0f} min")
        if not printed_scale:
            describe_preds(preds, tag=vid)      # confirm z-scored/signed scale ONCE
            printed_scale = True

        global_mag, lanes = arc_from_preds_multi(preds, LANE_MASKS)
        T = len(global_mag)
        lane_names = list(lanes)                 # column order, e.g. ['dan','language']
        with open(OUT_DIR / f"arc_{vid}.csv", "w") as f:
            f.write("t_sec,global_mag" + "".join(f",{n}_mag" for n in lane_names) + "\n")
            for t in range(T):
                f.write(f"{t},{global_mag[t]:.6f}"
                        + "".join(f",{lanes[n][t]:.6f}" for n in lane_names) + "\n")

        # arc_<id>.json shows DEMO_FEATURE's lane if present, else whole-cortex global
        demo_arc = lanes[DEMO_FEATURE] if DEMO_FEATURE in lanes else global_mag
        feat = DEMO_FEATURE if DEMO_FEATURE in lanes else "global"
        spots = detect_weak_spots(demo_arc, fps=1.0)
        write_demo_json(OUT_DIR, vid, demo_arc, spots, feature_name=feat)
        print(f"  ok: T={T}s  weak_spots={len(spots)}  feature={feat}  cols={['global']+lane_names}")
        n_ok += 1
    except Exception as e:   # one bad clip must not kill the batch
        n_fail += 1
        print(f"  [ERROR] {vid}: {e!r} - skipping, batch continues")
        traceback.print_exc()
    # free VRAM + host RAM between clips. The COGNIMUSE films are 30+ min (3-6x longer than
    # any TVSum clip this notebook was tested on), so release cached allocations each iteration
    # to guard against a slow OOM partway through the batch. Guarded so it can never break a run.
    try:
        import torch, gc
        del preds
        gc.collect()
        torch.cuda.empty_cache()
    except Exception:
        pass

print(f"\n[batch done] ok={n_ok} failed={n_fail}  ->  {OUT_DIR}")
print("Reminder: preds are z-scored SIGNED BOLD (~[-1,1]), NOT probabilities - "
      "confirm from the [preds ...] distribution printed above.")


## Cell 5 — zip the results and download

In [ ]:
# === Cell 5: zip + download ======================================================
import shutil
from pathlib import Path

archive = shutil.make_archive("/content/soma_arcs", "zip", root_dir=str(OUT_DIR))
print("zipped ->", archive)

try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("Auto-download unavailable:", e)
    print(f"Your results are already saved in Drive at: {OUT_DIR}")


# TRIMODAL (audio + video + **text / dialogue**) — the FULL TRIBE model

The cells above run **audio + video**. But the shipped `facebook/tribev2` checkpoint is actually
**trimodal**: its default `features_to_use` is `["text","audio","video"]`, and the AV cells
override it *down*. These cells run it at **full strength** — adding the dialogue/**text** branch
(whisperx transcript → **LLaMA-3.2-3B** embeddings), which is the exact configuration TRIBE
won **Algonauts-2025** with. It should predict the brain more faithfully.

### Prerequisites (all three)
1. **Hardware — A100 + High-RAM on.** Trimodal needs ~28–32 GB VRAM (LLaMA-3.2-3B + w2v-BERT + V-JEPA2-giant). **T4/L4 will OOM.**
2. **Gated model — `meta-llama/Llama-3.2-3B`** (the *base* model, **not** `-Instruct`). Request access at <https://huggingface.co/meta-llama/Llama-3.2-3B> (usually granted in minutes–hours), and use a token that has it. **Cell T1** logs you in and verifies this for you.
3. **whisperx** — installed by **Cell T1** (`uv` → `uvx whisperx`). The text branch transcribes the dialogue itself; **you provide no transcript.**

### Run order (fresh session)
**Cell 1** (install → **restart**) → **Cell T1** (trimodal deps + HF login; wait for `[OK]`) → **Cell 2B** (mount Drive + discover clips) → **Cell 2C** (build the DAN + LANGUAGE masks) → **Cell T3** (load trimodal model) → **Cell T4** (extract). *(Skip Cell 2 and Cells 3–4 — those are the AV path.)*

### What it writes
A **separate** `MyDrive/soma/arcs_trimodal/` folder (it **never** overwrites your AV `arcs/`), same `preds_<id>.npy` / `arc_<id>.csv` (`t_sec,global_mag,dan_mag,language_mag`) / `arc_<id>.json` schema — so you can line up **av vs trimodal for the same clip** and drop both into the identical analysis. **This is the run that makes `language_mag` the real MESSAGE lane:** the text branch drives the language cortex, whereas on the AV path that column is only a shape-valid plumbing demo. Resume/skip-cached works exactly like Cell 4.

### Honesty (unchanged)
The validated claim is still only *video/audio/text → brain activation*. Trimodal is expected to predict the **brain** more faithfully (it's the full model), but whether that improves our downstream **attention/affect** correlation is its **own** empirical question. **Report av vs trimodal side by side — a tie, or trimodal losing, is a real, publishable result. Do not cherry-pick the better one.**

In [ ]:
# === Cell T1 — trimodal prereqs: whisperx (via uv) + gated LLaMA-3.2-3B login =====
# Run AFTER Cell 1's restart, in the SAME session as T3/T4. These installs DON'T touch
# numpy/scipy, so there is NO second restart. The text branch needs two things:
#   1) `uvx whisperx` — tribev2 transcribes the dialogue by shelling out to `uvx whisperx`,
#      so `uv` must be on PATH. We install it + pre-warm whisperx (first clip then doesn't
#      stall). You provide no transcript; it makes its own.
#   2) meta-llama/Llama-3.2-3B (GATED) — the text embedder. We log in + verify access NOW
#      so a missing grant fails here (fast), not 40 min into a batch.
import subprocess, sys, shutil, os

# 1) uv -> provides `uvx`
if shutil.which("uvx") is None:
    print("Installing uv (provides `uvx` for whisperx) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
print("uvx ->", shutil.which("uvx") or "STILL MISSING (rerun this cell)")

# 2) pre-warm whisperx into a persistent uv tool env (one-time; the large-v3 + wav2vec2
#    model weights still download on the first REAL clip). Non-fatal: if this errors,
#    `uvx whisperx` falls back to an on-demand install at first use.
print("Pre-warming whisperx (one-time, a few min — pip noise is normal) ...")
subprocess.run(["uv", "tool", "install", "-q", "whisperx"], check=False)
_wx = subprocess.run(["uvx", "whisperx", "--help"], capture_output=True, text=True)
print("whisperx reachable via uvx:",
      "YES" if _wx.returncode == 0 else f"NO -> {(_wx.stderr or '')[-300:]}")

# 3) HuggingFace login + gated-access pre-flight for meta-llama/Llama-3.2-3B (the base model).
#    Token source order:  HF_TOKEN env  ->  Colab Secret named HF_TOKEN  ->  interactive paste.
#    A Colab Secret SURVIVES the Cell-1 restart, so set it ONCE instead of re-pasting each run:
#      left sidebar 🔑 (Secrets) -> + Add new secret -> name = HF_TOKEN, value = your hf_... token
#      -> toggle "Notebook access" ON.  (Your laptop's .env is NOT visible to Colab — copy the
#      token from there into this secret.)
from huggingface_hub import login
_tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not _tok:
    try:
        from google.colab import userdata
        _tok = userdata.get("HF_TOKEN")
    except Exception:
        _tok = None
if _tok:
    login(token=_tok, add_to_git_credential=False)
    print("Logged in via HF_TOKEN secret/env (no paste needed).")
else:
    print("\nNo HF_TOKEN secret found. Either add one (🔑 sidebar, name it HF_TOKEN) and re-run,")
    print("or paste a token with meta-llama/Llama-3.2-3B access now "
          "(https://huggingface.co/settings/tokens):")
    login()   # interactive widget; stores the token for THIS session only

def _has_llama_access(repo="meta-llama/Llama-3.2-3B"):
    try:
        from huggingface_hub import auth_check
    except Exception:
        auth_check = None
    if auth_check is not None:
        try:
            auth_check(repo); return True, None
        except Exception as e:
            return False, e
    try:  # older hub without auth_check: probe a small gated file
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo, "config.json"); return True, None
    except Exception as e:
        return False, e

ok, err = _has_llama_access()
if ok:
    print("\n[OK] Llama-3.2-3B access confirmed — trimodal is good to go.")
    print("     Next: Cell 2B (mount Drive + clips) -> Cell T3 (load) -> Cell T4 (extract).")
else:
    print(f"\n[GATED / NO ACCESS] meta-llama/Llama-3.2-3B is not reachable yet ({err!r}).")
    print("  -> confirm the grant at https://huggingface.co/meta-llama/Llama-3.2-3B, make sure")
    print("     your token can read it, then RE-RUN this cell. Do NOT run T3/T4 until this [OK].")


In [ ]:
# === Cell T3 — load TRIBE v2 in TRIMODAL config (audio + video + TEXT) ============
# Same as Cell 3, but features_to_use ALSO includes "text" (which is the shipped default —
# the AV cells override it OFF). Loaded as `model_tri` so it never clobbers an AV `model`.
# Run only AFTER Cell T1 prints "Llama-3.2-3B access confirmed".
import os
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
from pathlib import Path
from tribev2.demo_utils import TribeModel

NUM_WORKERS = 2   # Colab has ~2 cores; the shipped default (20) thrashes them. Changes NO numbers.

# Cache on Drive (mounted by Cell 2B) so the ~1 GB brain weights + the LLaMA/w2v/vjepa
# feature caches SURVIVE a disconnect and never re-download.
CACHE_FOLDER = (Path("/content/drive/MyDrive/soma/cache")
                if Path("/content/drive/MyDrive").exists() else Path("./cache"))
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"cache_folder = {CACHE_FOLDER}")

print("Loading facebook/tribev2 (audio+video+TEXT) — the first trimodal PREDICT also pulls "
      "LLaMA-3.2-3B (~6 GB) + w2v-BERT on top of the AV weights ...")
model_tri = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={"data": {"features_to_use": ["audio", "video", "text"],
                            "num_workers": NUM_WORKERS}},
)

# Same DataLoader worker cap as Cell 3 (idempotent — safe even if Cell 3 already patched it).
import torch
if not getattr(torch.utils.data.DataLoader.__init__, "_soma_capped", False):
    _orig_dl_init = torch.utils.data.DataLoader.__init__
    def _capped_dl_init(self, *a, num_workers=0, **k):
        _orig_dl_init(self, *a, num_workers=min(num_workers or 0, NUM_WORKERS), **k)
    _capped_dl_init._soma_capped = True
    torch.utils.data.DataLoader.__init__ = _capped_dl_init
    print(f"[speed] DataLoader workers capped at {NUM_WORKERS}.")

print("Trimodal model loaded.  features_to_use = ['audio','video','text']  (LLaMA-3.2-3B ON).")


In [ ]:
# === Cell T4 — TRIMODAL extract -> preds/arc into a SEPARATE arcs_trimodal/ ========
# Reuses the clips + ROI mask discovered by Cell 2B and the model_tri from Cell T3, and
# writes to OUT_DIR_TRI = <your arcs>_trimodal so AV and trimodal outputs NEVER mix.
# Self-contained (re-defines the small reduce helpers) so it runs without Cell 4.
import json, time, traceback
import numpy as np
import pandas as pd
from pathlib import Path

# --- guards ----------------------------------------------------------------------
for _v, _msg in [("clips", "run Cell 2B (discovers clips + ROI mask)"),
                 ("model_tri", "run Cell T3 (loads the trimodal model)")]:
    if _v not in globals():
        raise SystemExit(f"→ Missing `{_v}` — {_msg} first.")
if not clips:
    raise SystemExit("0 clips — upload clips + run Cell 2B first.")

# separate output dir so trimodal never overwrites the AV arcs (compare av vs trimodal)
OUT_DIR_TRI = OUT_DIR.parent / (OUT_DIR.name + "_trimodal")
OUT_DIR_TRI.mkdir(parents=True, exist_ok=True)
print(f"trimodal out -> {OUT_DIR_TRI}   (AV arcs in {OUT_DIR} are left untouched)")

# The reference TRIMODAL event build IS tribev2's own get_audio_and_text_events(audio_only=False):
#   ExtractAudioFromVideo -> ChunkEvents(Audio) -> ChunkEvents(Video)
#   -> ExtractWordsFromAudio (whisperx transcript) -> AddText -> AddSentenceToWords
#   -> AddContextToWords -> RemoveMissing   (the "Word" events the LLaMA text branch reads)
from tribev2.demo_utils import get_audio_and_text_events
def build_events_trimodal(video_path):
    df = pd.DataFrame([{"type": "Video", "filepath": str(video_path), "start": 0,
                        "timeline": "default", "subject": "default"}])
    return get_audio_and_text_events(df, audio_only=False)

# --- helpers (mirror Cell 4 / batch_extract.py) ----------------------------------
def describe_preds(preds, tag=""):
    p = np.asarray(preds, float); pct = np.percentile(p, [1, 50, 90, 99])
    print(f"  [preds {tag}] shape={p.shape} min={p.min():.3f} max={p.max():.3f} "
          f"mean={p.mean():.3f} std={p.std():.3f}")
    print(f"           pct(1/50/90/99)={pct[0]:.3f}/{pct[1]:.3f}/{pct[2]:.3f}/{pct[3]:.3f}"
          f"  frac<0={np.mean(p < 0):.2%}")
    if p.min() >= 0 and p.max() <= 1:
        print("           WARNING: looks bounded [0,1] — verify it's z-scored BOLD, not a prob.")

def arc_from_preds(preds, roi_mask=None):
    p = np.abs(np.asarray(preds, float)); g = p.mean(axis=1); r = None
    if roi_mask is not None:
        m = np.asarray(roi_mask, bool)
        if m.shape[0] != p.shape[1]:
            raise ValueError(f"ROI mask length {m.shape[0]} != n_vertices {p.shape[1]}")
        r = p[:, m].mean(axis=1)
    return g, r

def arc_from_preds_multi(preds, lane_masks):
    p = np.abs(np.asarray(preds, float)); g = p.mean(axis=1); lanes = {}
    for name, m in lane_masks.items():
        m = np.asarray(m, bool)
        if m.shape[0] != p.shape[1]:
            raise ValueError(f"lane {name!r} mask length {m.shape[0]} != n_vertices {p.shape[1]}")
        lanes[name] = p[:, m].mean(axis=1)
    return g, lanes

def detect_weak_spots(arc, fps=1.0, min_len_sec=3, drop_pctl=25):
    a = np.asarray(arc, float)
    if len(a) < 5: return []
    k = max(1, int(round(fps))); sm = np.convolve(a, np.ones(k) / k, mode="same")
    thr = np.percentile(sm, drop_pctl); low = sm <= thr
    spots, i, n = [], 0, len(a); min_len = int(round(min_len_sec * fps))
    while i < n:
        if low[i]:
            j = i
            while j < n and low[j]: j += 1
            if (j - i) >= min_len:
                spots.append({"start": round(i / fps, 2), "end": round(j / fps, 2),
                              "label": f"predicted dip - {round((j - i) / fps)}s low-activation stretch"})
            i = j
        else: i += 1
    return spots

def write_demo_json(out_dir, vid, arc, weak_spots, feature_name, fps=1.0):
    a = np.asarray(arc, float); lo, hi = np.percentile(a, 2), np.percentile(a, 98)
    norm = np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
    payload = {"video_id": vid, "fps_arc": fps, "duration_sec": round(len(a) / fps, 2),
               "feature": feature_name, "precomputed": True, "modality": "trimodal",
               "claim": {"validated": "video+audio+text -> brain activation (Meta TRIBE v2, "
                                      "public model, benchmarked vs real fMRI)",
                         "hypothesis": "activation -> engagement (our downstream inference; "
                                       "tested, not asserted)"},
               "timestamps": [round(i / fps, 2) for i in range(len(a))],
               "activation": [round(float(x), 4) for x in norm], "weak_spots": weak_spots}
    (out_dir / f"arc_{vid}.json").write_text(json.dumps(payload, indent=2))

# --- LANE masks (reuse the ones Cell 2C built / Cell 2B resolved) -----------------
LANE_MASKS = {}
_lane_paths = {}
if "ROI_MASKS" in globals() and ROI_MASKS:
    _lane_paths = dict(ROI_MASKS)
elif "ROI_MASK_PATH" in globals() and ROI_MASK_PATH is not None:
    _lane_paths = {"roi": ROI_MASK_PATH}
for _name, _p in _lane_paths.items():
    _m = np.asarray(np.load(_p), bool)
    LANE_MASKS[_name] = _m
    print(f"[lane] {_name:9s} {Path(_p).name}: {int(_m.sum())}/{_m.shape[0]} vertices")
if not LANE_MASKS:
    print("[note] no lane masks -> GLOBAL only. Run Cell 2C first for dan_mag + language_mag.")

# --- batch (resume vs OUT_DIR_TRI; one bad clip cannot kill the batch) ------------
done = {p.stem[len("preds_"):] for p in OUT_DIR_TRI.glob("preds_*.npy")}
todo = [c for c in clips if c.stem not in done]
print(f"{len(clips)} clips | {len(done)} trimodal-done -> {len(todo)} to run\n")
printed_scale = False; n_ok = n_fail = 0; clip_secs = []; _left = len(todo)
for vp in clips:
    vid = vp.stem; npy = OUT_DIR_TRI / f"preds_{vid}.npy"
    try:
        if npy.exists():
            print(f"[skip-cached] {vid}"); preds = np.load(npy)
        else:
            print(f"[predict-trimodal] {vid} ... ({_left} to go)  transcribe -> embed -> brain")
            _t0 = time.time()
            events = build_events_trimodal(vp)
            preds, _seg = model_tri.predict(events=events)
            preds = np.asarray(preds, float)
            np.save(npy, preds.astype(np.float32))
            _dt = time.time() - _t0; clip_secs.append(_dt); _left -= 1
            _avg = sum(clip_secs) / len(clip_secs)
            print(f"    took {_dt:.0f}s | avg {_avg:.0f}s/clip | ETA {_left}: ~{_avg * _left / 60:.0f} min")
        if not printed_scale:
            describe_preds(preds, tag=vid); printed_scale = True
        g, lanes = arc_from_preds_multi(preds, LANE_MASKS); T = len(g)
        lane_names = list(lanes)
        with open(OUT_DIR_TRI / f"arc_{vid}.csv", "w") as f:
            f.write("t_sec,global_mag" + "".join(f",{n}_mag" for n in lane_names) + "\n")
            for t in range(T):
                f.write(f"{t},{g[t]:.6f}" + "".join(f",{lanes[n][t]:.6f}" for n in lane_names) + "\n")
        demo_arc = lanes[DEMO_FEATURE] if DEMO_FEATURE in lanes else g
        feat = DEMO_FEATURE if DEMO_FEATURE in lanes else "global"
        spots = detect_weak_spots(demo_arc)
        write_demo_json(OUT_DIR_TRI, vid, demo_arc, spots, feature_name=feat)
        print(f"  ok: T={T}s  weak_spots={len(spots)}  feature={feat}  cols={['global']+lane_names}"); n_ok += 1
    except Exception as e:   # one bad clip (e.g. whisperx failure) must not kill the batch
        n_fail += 1
        print(f"  [ERROR] {vid}: {e!r} — skipping, batch continues"); traceback.print_exc()
    try:
        import torch, gc; del preds; gc.collect(); torch.cuda.empty_cache()
    except Exception: pass

print(f"\n[trimodal batch done] ok={n_ok} failed={n_fail} -> {OUT_DIR_TRI}")
print("Compare vs AV: same <id>, arcs/ (av) vs arcs_trimodal/ (trimodal). preds are "
      "z-scored SIGNED BOLD (~[-1,1]), NOT probabilities.")
